In [1]:
import pandas as pd
import sqlite3

In [2]:
claims = pd.read_csv("claims_raw.csv")
customers = pd.read_csv("customers.csv")

In [3]:
print("Claims:", claims.shape)
print("Customers:", customers.shape)

Claims: (8045, 14)
Customers: (2500, 4)


In [4]:
conn = sqlite3.connect("insurance_claims.db")

In [5]:
claims.to_sql("claims", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)

2500

In [6]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query, conn)

,name
0,policies
1,claims_clean
2,claims_analysis
3,claims
4,customers



Validate the number of claim records and unique Claim IDs before analysis.

In [7]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Claim_ID) AS unique_claims
FROM claims;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_claims
0,8045,8000


Total 8,045 records; 8,000 unique Claim IDs; 45 duplicate claim records. 

Create a cleaned claims table containing one record per unique Claim ID.

In [8]:
query = """
CREATE TABLE claims_clean AS
SELECT *
FROM claims
WHERE rowid IN (
    SELECT MIN(rowid)   
    FROM claims
    GROUP BY Claim_ID
);
"""

conn.execute("DROP TABLE IF EXISTS claims_clean")
conn.execute(query)
conn.commit()

In [9]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Claim_ID) AS unique_claims
FROM claims_clean;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_claims
0,8000,8000


In [10]:
query = """
SELECT
    SUM(CASE WHEN Claim_Amount IS NULL THEN 1 ELSE 0 END) AS missing_claim_amount,
    SUM(CASE WHEN Region IS NULL THEN 1 ELSE 0 END) AS missing_region,
    SUM(CASE WHEN Settlement_Date IS NULL THEN 1 ELSE 0 END) AS missing_settlement_date
FROM claims_clean;
"""

pd.read_sql_query(query, conn)

,missing_claim_amount,missing_region,missing_settlement_date
0,55,70,1968


In [11]:
query = """
SELECT
    Claim_Status,
    COUNT(*) AS missing_settlement_dates
FROM claims_clean
WHERE Settlement_Date IS NULL
GROUP BY Claim_Status
ORDER BY missing_settlement_dates DESC;
"""

pd.read_sql_query(query, conn)

,Claim_Status,missing_settlement_dates
0,Pending,1125
1,Under Review,843


In [12]:
query = """
SELECT
    c.Claim_ID,
    c.Customer_ID,
    c.Region AS claim_region,
    cu.Customer_Region
FROM claims_clean c
LEFT JOIN customers cu
    ON c.Customer_ID = cu.Customer_ID
WHERE c.Region IS NULL;
"""

pd.read_sql_query(query, conn)

,Claim_ID,Customer_ID,claim_region,Customer_Region
0,CLM000139,CUST01360,None,North
1,CLM000171,CUST02400,None,West
2,CLM000239,CUST00113,None,East
3,CLM000270,CUST01710,None,North
4,CLM000332,CUST01652,None,East
...,...,...,...,...
65,CLM007316,CUST01975,None,East
66,CLM007331,CUST02142,None,North
67,CLM007371,CUST02436,None,North
68,CLM007391,CUST00309,None,North


In [13]:
query = """
CREATE TABLE claims_analysis AS
SELECT
    c.Claim_ID,
    c.Customer_ID,
    c.Policy_ID,

    CASE
    WHEN LOWER(TRIM(c.Claim_Type)) IN ('hospitalisation', 'hospitalization')
        THEN 'Hospitalization'

    WHEN LOWER(TRIM(c.Claim_Type)) IN ('medical emerg.', 'medical emergency')
        THEN 'Medical Emergency'

    WHEN LOWER(TRIM(c.Claim_Type)) IN ('property dmg', 'property damage')
        THEN 'Property Damage'

    WHEN LOWER(TRIM(c.Claim_Type)) IN ('vehicle dmg', 'vehicle damage')
        THEN 'Vehicle Damage'

    ELSE TRIM(c.Claim_Type)
END AS Claim_Type,

    COALESCE(c.Region, cu.Customer_Region) AS Region,
    c.Claim_Amount,
    c.Claim_Date,
    c.Settlement_Date,
    c.Claim_Status,
    c.Processing_Days,
    c.SLA_Days,
    c.Agent_ID,
    c.Policy_Type,
    c.Workflow,

    CASE
        WHEN c.Processing_Days > c.SLA_Days THEN 'SLA Breach'
        ELSE 'Within SLA'
    END AS SLA_Status,

    CASE
        WHEN c.Processing_Days > c.SLA_Days
        THEN c.Processing_Days - c.SLA_Days
        ELSE 0
    END AS Delay_Days

FROM claims_clean c

LEFT JOIN customers cu
    ON c.Customer_ID = cu.Customer_ID;
"""

conn.execute("DROP TABLE IF EXISTS claims_analysis")
conn.execute(query)
conn.commit()

In [14]:
query = """
SELECT
    COUNT(*) AS total_claims,
    SUM(CASE WHEN Region IS NULL THEN 1 ELSE 0 END) AS missing_region,
    SUM(CASE WHEN SLA_Status = 'SLA Breach' THEN 1 ELSE 0 END) AS sla_breaches
FROM claims_analysis;
"""

pd.read_sql_query(query, conn)

,total_claims,missing_region,sla_breaches
0,8000,0,4025


In [15]:
query = """
SELECT
    Region,
    COUNT(*) AS total_claims,
    SUM(CASE
        WHEN SLA_Status = 'SLA Breach' THEN 1
        ELSE 0
    END) AS sla_breaches,
    ROUND(
        100.0 * SUM(CASE
            WHEN SLA_Status = 'SLA Breach' THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS breach_rate_pct,
    ROUND(AVG(Processing_Days), 2) AS avg_processing_days
FROM claims_analysis
GROUP BY Region
ORDER BY breach_rate_pct DESC;
"""

pd.read_sql_query(query, conn)

,Region,total_claims,sla_breaches,breach_rate_pct,avg_processing_days
0,East,1459,977,66.96,16.81
1,Central,1188,695,58.50,15.57
2,North,1718,823,47.90,14.18
3,West,1872,792,42.31,13.07
4,South,1763,738,41.86,13.15


In [16]:
query = """
SELECT
    Workflow,
    COUNT(*) AS total_claims,
    ROUND(AVG(Processing_Days), 2) AS avg_processing_days,
    SUM(CASE
        WHEN SLA_Status = 'SLA Breach' THEN 1
        ELSE 0
    END) AS sla_breaches,
    ROUND(
        100.0 * SUM(CASE
            WHEN SLA_Status = 'SLA Breach' THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS breach_rate_pct
FROM claims_analysis
GROUP BY Workflow
ORDER BY breach_rate_pct DESC;
"""

pd.read_sql_query(query, conn)

,Workflow,total_claims,avg_processing_days,sla_breaches,breach_rate_pct
0,Fraud Review,814,25.98,811,99.63
1,Manual Review,1627,19.12,1457,89.55
2,Standard,4332,11.95,1569,36.22
3,Fast Track,1227,8.99,188,15.32


In [17]:
query = """
SELECT
    Region,
    Workflow,
    COUNT(*) AS total_claims,
    SUM(CASE
        WHEN SLA_Status = 'SLA Breach' THEN 1
        ELSE 0
    END) AS sla_breaches,
    ROUND(
        100.0 * SUM(CASE
            WHEN SLA_Status = 'SLA Breach' THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS breach_rate_pct,
    ROUND(AVG(Processing_Days), 2) AS avg_processing_days
FROM claims_analysis
GROUP BY Region, Workflow
ORDER BY Region, breach_rate_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

,Region,Workflow,total_claims,sla_breaches,breach_rate_pct,avg_processing_days
0,Central,Fraud Review,127,127,100.00,27.46
1,Central,Manual Review,232,221,95.26,19.98
2,Central,Standard,628,302,48.09,13.29
3,Central,Fast Track,201,45,22.39,10.10
4,East,Fraud Review,130,130,100.00,28.26
5,East,Manual Review,298,290,97.32,21.83
6,East,Standard,814,492,60.44,14.59
7,East,Fast Track,217,65,29.95,11.41
8,North,Fraud Review,189,188,99.47,25.99
9,North,Manual Review,326,296,90.80,19.01


In [18]:
query = """
SELECT
    Region,
    COUNT(*) AS total_claims,

    SUM(CASE
        WHEN Workflow IN ('Manual Review', 'Fraud Review')
        THEN 1 ELSE 0
    END) AS review_claims,

    ROUND(
        100.0 * SUM(CASE
            WHEN Workflow IN ('Manual Review', 'Fraud Review')
            THEN 1 ELSE 0
        END) / COUNT(*),
        2
    ) AS review_workflow_pct

FROM claims_analysis
GROUP BY Region
ORDER BY review_workflow_pct DESC;
"""

pd.read_sql_query(query, conn)

,Region,total_claims,review_claims,review_workflow_pct
0,South,1763,557,31.59
1,West,1872,582,31.09
2,Central,1188,359,30.22
3,North,1718,515,29.98
4,East,1459,428,29.34


In [19]:
query = """
SELECT
    Region,
    Workflow,
    ROUND(AVG(Processing_Days), 2) AS avg_processing_days,
    ROUND(AVG(Delay_Days), 2) AS avg_delay_days,
    ROUND(
        100.0 * SUM(CASE
            WHEN SLA_Status = 'SLA Breach' THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS breach_rate_pct
FROM claims_analysis
GROUP BY Region, Workflow
ORDER BY Workflow, breach_rate_pct DESC;
"""

pd.read_sql_query(query, conn)

,Region,Workflow,avg_processing_days,avg_delay_days,breach_rate_pct
0,East,Fast Track,11.41,1.06,29.95
1,Central,Fast Track,10.10,0.68,22.39
2,North,Fast Track,8.82,0.41,12.20
3,West,Fast Track,7.68,0.26,10.55
4,South,Fast Track,7.77,0.19,6.43
5,Central,Fraud Review,27.46,14.75,100.00
6,East,Fraud Review,28.26,15.85,100.00
7,South,Fraud Review,24.59,11.98,100.00
8,North,Fraud Review,25.99,13.24,99.47
9,West,Fraud Review,24.72,11.69,98.92


In [20]:
query = """
SELECT
    Agent_ID,
    COUNT(*) AS total_claims,
    ROUND(AVG(Processing_Days),2) AS avg_processing_days,
    ROUND(AVG(Delay_Days),2) AS avg_delay_days

FROM claims_analysis

GROUP BY Agent_ID

HAVING COUNT(*) >= 50

ORDER BY avg_processing_days DESC

LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Agent_ID,total_claims,avg_processing_days,avg_delay_days
0,AGT052,100,19.64,6.97
1,AGT065,115,19.47,7.07
2,AGT078,98,19.03,6.52
3,AGT039,96,18.65,6.71
4,AGT013,100,18.55,6.52
5,AGT026,113,18.13,5.88
6,AGT061,98,15.83,4.28
7,AGT005,103,15.81,4.53
8,AGT012,97,15.60,3.91
9,AGT018,96,15.58,3.81


In [21]:
query = """
SELECT
    Claim_ID,
    Region,
    Workflow,
    Claim_Status,
    Processing_Days

FROM claims_analysis

WHERE Processing_Days >
(
    SELECT AVG(Processing_Days)
    FROM claims_analysis
)

ORDER BY Processing_Days DESC;
"""

pd.read_sql_query(query, conn)

,Claim_ID,Region,Workflow,Claim_Status,Processing_Days
0,CLM006593,North,Fraud Review,Settled,45
1,CLM003174,East,Fraud Review,Pending,43
2,CLM004662,East,Fraud Review,Pending,42
3,CLM001171,North,Fraud Review,Settled,41
4,CLM002063,East,Manual Review,Settled,41
...,...,...,...,...,...
3624,CLM007855,North,Standard,Settled,15
3625,CLM007872,South,Standard,Settled,15
3626,CLM007937,South,Standard,Settled,15
3627,CLM007980,West,Standard,Settled,15


# Key Findings
1. East showed the highest SLA breach rate (66.96%) and average processing time (16.81 days), followed by Central.
2. Fraud Review (99.63%) and Manual Review (89.55%) were the major SLA bottlenecks.
3. Workflow mix did not explain East's poor SLA performance; processing times were consistently higher across workflows, indicating region-level operational inefficiencies.
4. The findings suggest prioritizing process and review-capacity improvements in East and Central, particularly for review-intensive workflows.